In [31]:
import tensorflow as tf 
from tensorflow.keras.datasets import cifar10 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
import pandas as pd

In [32]:
(xtrain, ytrain), (xtest, ytest) = cifar10.load_data()
xtrain, xtest = xtrain/255.0, xtest/255.0
ytrain, ytest = to_categorical(ytrain), to_categorical(ytest)

In [33]:
def build_model(init, dropout, reg):
    model = Sequential([Flatten(input_shape=(32, 32, 3))])
    for units in [512, 256, 64]:
        model.add(Dense(units, activation="relu",
                        kernel_initializer=init,
                        kernel_regularizer=reg))
        if dropout:
            model.add(Dropout(0.5))
    model.add(Dense(10, activation="softmax", kernel_initializer=init))
    return model

In [34]:
experiments = [
    ("Baseline","glorot_uniform",False,None,"None"),
    ("Xavier","glorot_uniform",False,None,"None"),
    ("Kaiming","glorot_uniform",False,None,"None"),
    ("Dropout","glorot_uniform",True,None,"None"),
    ("L2 Regularization","glorot_uniform",False,l2(0.001),"L2")
]

In [35]:
results=[]

In [36]:
for name,init,dropout,reg,reg_label in experiments:
    for opt in ["Adam", "SGD"]:
        model = build_model(init, dropout, reg)
        model.compile(
            optimizer=opt.lower(),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )
        model.fit(xtrain, ytrain, epochs=5, batch_size=64, validation_data=(xtest, ytest))
        loss, acc = model.evaluate(xtest, ytest, verbose=0)
        results.append([name,opt,reg_label,loss,acc*100])
results_df = pd.DataFrame(results, columns=[
    "Model", "Optimizer", "Regularization",
    "Test Loss", "Test Accuracy (%)"
])

print("\n" + "=" * 100)
print("FINAL COMPARISON")
print("=" * 100)
print(results_df.to_string(index=False))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 56s 59ms/step - accuracy: 0.3189 - loss: 1.8828 - val_accuracy: 0.3443 - val_loss: 1.7764
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 81s 58ms/step - accuracy: 0.3917 - loss: 1.6910 - val_accuracy: 0.4127 - val_loss: 1.6320
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 101s 81ms/step - accuracy: 0.4264 - loss: 1.6010 - val_accuracy: 0.4239 - val_loss: 1.6092
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 62ms/step - accuracy: 0.4496 - loss: 1.5397 - val_accuracy: 0.4461 - val_loss: 1.5318
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 64s 82ms/step - accuracy: 0.4643 - loss: 1.4937 - val_accuracy: 0.4615 - val_loss: 1.5132
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.3042 - loss: 1.9331 - val_accuracy: 0.3089 - val_loss: 1.8792
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 39ms/step - accuracy: 0.3793 - loss: 1.7463 - val_accuracy: 0.3664 - val_loss: 1.7493
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - accuracy: 0.1729 - loss: 2.1217 - val_acc